# Text Generation

Text generation models predict the next token given a context, one token at a time. In this notebook we explore **GPT-2**, understand different **decoding strategies** (greedy, sampling, beam search), and see how parameters like `temperature`, `top_p`, and `top_k` control output creativity.

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers pillow -q

## Imports

In [ ]:
from transformers import pipeline, set_seed
import torch

device = 0 if torch.cuda.is_available() else -1
set_seed(42)  # for reproducible outputs
print(f"PyTorch {torch.__version__} | device: {'GPU' if device == 0 else 'CPU'}")

## 1. Basic Text Generation with GPT-2

In [ ]:
generator = pipeline("text-generation", model="gpt2", device=device)

prompt = "Artificial intelligence will"
result = generator(prompt, max_new_tokens=40, do_sample=False)  # greedy decoding
print(result[0]["generated_text"])

## 2. Decoding Strategies

| Strategy | Description | Best for |
|---|---|---|
| **Greedy** | Always pick the highest-probability next token | Deterministic, factual tasks |
| **Beam Search** | Keep top-k sequences at each step, pick best overall | Translation, summarization |
| **Sampling** | Sample from the probability distribution | Creative, diverse outputs |
| **Top-k Sampling** | Sample only from the top-k most likely tokens | Balance creativity/coherence |
| **Top-p (Nucleus)** | Sample from smallest set of tokens summing to probability p | State-of-the-art creative generation |

In [ ]:
prompt = "The future of robotics is"

print("=== GREEDY (deterministic) ===")
out = generator(prompt, max_new_tokens=30, do_sample=False)
print(out[0]["generated_text"])

print("\n=== BEAM SEARCH (num_beams=5) ===")
out = generator(prompt, max_new_tokens=30, num_beams=5, do_sample=False, early_stopping=True)
print(out[0]["generated_text"])

print("\n=== SAMPLING (random) ===")
set_seed(0)
out = generator(prompt, max_new_tokens=30, do_sample=True)
print(out[0]["generated_text"])

## 3. Temperature — Controlling Creativity

`temperature` scales the logits before sampling. 
- **Low temperature (< 1.0):** sharper distribution → more predictable, repetitive text
- **High temperature (> 1.0):** flatter distribution → more surprising, potentially incoherent text
- **temperature = 1.0:** original distribution (default)

In [ ]:
prompt = "Once upon a time in a land far away"

for temp in [0.2, 0.7, 1.0, 1.5]:
    set_seed(42)
    out = generator(prompt, max_new_tokens=30, do_sample=True, temperature=temp)
    text = out[0]["generated_text"][len(prompt):]
    print(f"temp={temp:.1f}: ...{text.strip()}")

## 4. Top-k and Top-p (Nucleus) Sampling

- **`top_k=50`:** at each step, keep only the 50 most probable tokens and sample from them
- **`top_p=0.9`:** keep the smallest set of tokens whose cumulative probability ≥ 0.9

These are often combined: `do_sample=True, top_k=50, top_p=0.95, temperature=0.8`

In [ ]:
prompt = "Scientists recently discovered"

configs = [
    {"label": "top_k=10",                 "top_k": 10,  "top_p": 1.0, "temperature": 1.0},
    {"label": "top_k=50",                 "top_k": 50,  "top_p": 1.0, "temperature": 1.0},
    {"label": "top_p=0.9",                "top_k": 0,   "top_p": 0.9, "temperature": 1.0},
    {"label": "top_k=50 + top_p=0.95 + temp=0.8", "top_k": 50, "top_p": 0.95, "temperature": 0.8},
]

for cfg in configs:
    set_seed(42)
    out = generator(
        prompt, max_new_tokens=25, do_sample=True,
        top_k=cfg["top_k"], top_p=cfg["top_p"], temperature=cfg["temperature"]
    )
    text = out[0]["generated_text"][len(prompt):]
    print(f"[{cfg['label']}]\n  ...{text.strip()}\n")

## 5. Repetition Penalty

Without repetition penalty, GPT-2 often gets stuck in loops. `repetition_penalty > 1.0` discourages repeating the same tokens.

In [ ]:
prompt = "The cat sat on the mat. The cat"

for rep_penalty in [1.0, 1.2, 1.5]:
    set_seed(42)
    out = generator(prompt, max_new_tokens=30, do_sample=False, repetition_penalty=rep_penalty)
    text = out[0]["generated_text"][len(prompt):]
    print(f"rep_penalty={rep_penalty}: ...{text.strip()}")

## 6. Prompt Engineering

The prompt is your primary tool to guide the model. Small changes in phrasing can dramatically change the output style and content.

In [ ]:
prompts = [
    "Machine learning is",                          # factual/neutral
    "Machine learning is exciting because",          # positive framing
    "The main problem with machine learning is",     # critical framing
    "In simple terms, machine learning means",       # explanatory framing
]

set_seed(42)
for p in prompts:
    out = generator(p, max_new_tokens=25, do_sample=True, top_p=0.9, temperature=0.8)
    continuation = out[0]["generated_text"][len(p):]
    print(f"Prompt : {p}")
    print(f"Output : {continuation.strip()}\n")